# Lab 1 — Convolution and classical filtering

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kleinric/cv-labs/blob/main/lab-01.ipynb)

**COMS4036A / COMS7050A Computer Vision · Week 1**

**Due: Thursday 30 July 2026, 17:00.** Groups of up to three; one member submits the group's notebook (`.ipynb`) on Moodle. Every member must be able to explain every cell.

Companion reading is [Chapter 1 of the course book](https://courses.ms.wits.ac.za/~richard/cv/book/). The chapter's widgets run most of this lab's operations interactively — use them to check your intuition against your code.

The Friday session covers Sections 0–5; Sections 6–8 are the take-home half. Labs are marked for completeness, correctness, and presentation: a notebook that runs top to bottom, with each written answer in its marked cell.

## 0. Working in Colab

This notebook is designed for [Google Colab](https://colab.research.google.com). Open it with the badge above — or download it from the book and **File ▸ Upload notebook**. Then, before anything else, **File ▸ Save a copy in Drive** — otherwise your work is not being saved.

Colab gives you a machine in the cloud with Python and the scientific stack preinstalled. Two things about it to learn today:

**Runtimes.** Under **Runtime ▸ Change runtime type** you choose the hardware. Everything in this lab runs on the CPU; leave the runtime on CPU so you are not queueing for a GPU you will not use. From Lab 2 onward the GPU matters, and this menu is where you will claim one.

**State.** Cells execute in the order *you run them*, not the order they appear, and variables live in the kernel between runs. A notebook that works because of a cell you ran three edits ago and then deleted is broken — you just cannot see it yet. The fix is a habit: **Runtime ▸ Restart session and run all**, then read every cell's output. Do this before you submit; a notebook that does not run top to bottom loses marks.

In [ ]:
# Group members — fill in before submitting.
MEMBERS = [
    # ("Student name", "Student number"),
    ( "Nkosenhle Ndlovu", "2539199" )
]

for name, number in MEMBERS:
    print(f"{number}  {name}")

In [ ]:
# Where is this running? On a CPU runtime, nvidia-smi fails — that is the
# expected output today.
!nvidia-smi

In [ ]:
import sys
import numpy as np
print(sys.version)
print("numpy", np.__version__)

## 1. Setup

The lab runs on the same images as the book: the dog-and-ball scene that every Chapter 1 figure uses, and the bark and grass textures from the texton section.

In [ ]:
BASE = "https://courses.ms.wits.ac.za/~richard/cv/book/widgets/assets"
!wget -q {BASE}/scene-480.jpg {BASE}/bark.png {BASE}/grass.png
!ls -l scene-480.jpg bark.png grass.png

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage
from skimage import color, io

plt.rcParams["figure.dpi"] = 90


def show(*images, titles=None, cmap="gray", size=3.2):
    """Display images side by side. Grayscale images are shown with
    vmin=0, vmax=1 — without that, imshow rescales each image to its own
    range and two images become impossible to compare."""
    fig, axes = plt.subplots(1, len(images), figsize=(size * len(images), size))
    axes = np.atleast_1d(axes)
    for k, (ax, im) in enumerate(zip(axes, images)):
        if im.ndim == 2:
            ax.imshow(np.clip(im, 0, 1), cmap=cmap, vmin=0, vmax=1)
        else:
            ax.imshow(im)
        ax.set_axis_off()
        if titles:
            ax.set_title(titles[k], fontsize=9)
    plt.tight_layout()
    plt.show()

## 2. An image is an array

Load the scene and look at what `imread` actually hands you.

In [ ]:
img = io.imread("scene-480.jpg")
print(img.shape, img.dtype, img.min(), img.max())
show(img)

**Q2.1.** From the shape alone: which axis is rows, which is columns, which is colour? Print the pixel at row 100, column 200, and say which of its three numbers is the red value.

In [ ]:
# YOUR CODE HERE
# numpy arrays are zero-indexed so that is why we chose to go with 99 and 199 instead of 100 and 200
px = img[99, 199]

print(px)

*Answer:* Because `img` has shape `(322, 480, 3)`, the first axis is rows, the second axis is columns, and the third axis is the colour channel. The pixel at row 100, column 200 is therefore `img[99, 199]` (zero-based indexing). Its values are `array([133, 121,  95], dtype=uint8)`, and the red value is the first entry, `133`.

---

**Q2.2.** Using slicing only — no loops — display the top-left quarter of the image, and the green channel of the full image as a grayscale picture.


In [ ]:
# YOUR CODE HERE

# half the image height and width
h_rows = img.shape[0] // 2
h_cols = img.shape[1] // 2

# top-left quarter: rows 0:h_rows, columns 0:h_cols
top_left_img = img[0:h_rows, 0:h_cols]

# green channel as a grayscale image (since array is zero-indexed, channel 1 is green)
green_channel = img[:, :, 1]

show(top_left_img, green_channel, titles=["Top-left quarter", "Green channel"])


### The `uint8` trap

The image arrived as `uint8`: integers 0–255, and arithmetic that wraps around silently.

In [ ]:
print(np.uint8(200) + np.uint8(100))

**Q2.3.** Brighten the image by adding 60 — directly on the `uint8` array — and display the result. Describe what you see and explain it. Then count how many values in the array wrapped around.

In [ ]:
# YOUR CODE HERE

brightened_image = img + np.uint8(60)

show(brightened_image)

# compare original against the brightened img index by index
wrap_count = np.count_nonzero(brightened_image < img)
print("Wrapped values:", wrap_count)


*Answer:* Adding 60 directly to the `uint8` image causes a wrap-around because `uint8` values are stored modulo 256 (8-bit slots in RAM). Basically, this means that the lowest value will always be 0 and the highest 255 and any arithmetic done results in values in this range.  Values near 255 overflow to small values, so the brightened image shows darker (or off-colour) bits at 70,352 pixels. That is 70,352 values wrapped around.

---

This bug is why the first thing done to any image in this course is a cast:


In [ ]:
imgf = img.astype(np.float64) / 255.0   # floats in [0, 1]
gray = color.rgb2gray(img)              # grayscale, also floats in [0, 1]
print(imgf.dtype, imgf.min(), imgf.max(), "|", gray.shape)

**Q2.4.** Redo the brightening on `imgf` (add 60/255). Floats do not wrap — but some values now exceed 1. What does `show` do about them? Then check: is this brightening reversible — does subtracting 60/255 recover the original exactly (`np.allclose`)? Contrast that with Q2.3, and say where information *would* be lost if you saved this brightened image to an ordinary 8-bit file. Compare the histograms of the image before and after using `plt.hist(x.ravel(), bins=256)`.

In [ ]:
# YOUR CODE HERE

# since imgf is in [0, 1] we cast everything to float by dividing by 255
brightened_imgf = imgf + (60 / 255.0)

# here we expect the same value as attained above in the uint8 demo (70,352)
exceeded_count = np.count_nonzero(brightened_imgf > 1.0)
print(exceeded_count, "values above 1.0 in brightened float image")

# `show` clips values outside [0, 1] to this range
show(imgf, brightened_imgf, titles=["Original float image", "Brightened float image"])

recovered = brightened_imgf - (60 / 255.0)
print("Is reversible?", np.allclose(recovered, imgf))
print("Max absolute difference:", np.max(np.abs(recovered - imgf)))

# here we display the histograms
plt.figure()
plt.hist(imgf.ravel(), bins=256, alpha=0.5, label="before") # slight transparency considered in alpha=0.5
plt.hist(brightened_imgf.ravel(), bins=256, alpha=0.5, label="after")
plt.legend()
plt.show()


*Answer:* In the float version, there is no wraparound because the values are stored as real numbers (i.e. floats). Examining the `show()` code, we observe that the 2-dimensional images will be clipped, by default to [0,1] range. Now, since our images are 3-dimensional they are displayed directly via `matplotlib.plot`'s `imshow()` function which (as their docs describe) clip RGB images to [0,1] if `float` and [0,255] if `int`.

Since the image is now in `float` the brightening is reversible: subtracting `60/255` from the brightened image recovers the original image exactly up to floating-point roundoff, so `np.allclose` is `True`. The round-off errors are purely computer and library induced which means in theory, in a perfect world, the round-off error is 0. 

This differs from Q2.3, where the `uint8` arithmetic wrapped around. If you saved the brightened float image to an ordinary 8-bit file, values above `1.0` would be clipped to `255`, so information in the brightest pixels would be lost due to the capping.


*Answer:*

## 3. Point operations

A point operation is a function applied to each pixel value independently: $v \mapsto f(v)$. Brightness is where the histogram sits — the mean $\mu$; contrast is how spread it is — the standard deviation $\sigma$. Each of these is a NumPy one-liner on `gray`.

**Q3.1.** Compute $\mu$ and $\sigma$ of `gray`. Then implement each of the following, display the result beside the original, and report the new $\mu$ and $\sigma$:

- brightness shift: $f(v) = v + b$ with $b = 0.2$
- contrast about the midpoint: $f(v) = \alpha(v - 0.5) + 0.5$ with $\alpha = 1.8$
- gamma: $f(v) = v^{1/2.2}$

In [ ]:
# YOUR CODE HERE

# original `gray` mean and std dev
gray_mean = np.mean(gray)
gray_std = np.std(gray)
print("gray mean/std:", gray_mean, gray_std)

# point operations on `gray`
brightness_shift = np.clip(gray + 0.2, 0.0, 1.0)
contrast_shift = np.clip(1.8 * (gray - 0.5) + 0.5, 0.0, 1.0)
gamma = np.clip(gray ** (1 / 2.2), 0.0, 1.0)

 # display
show(gray, brightness_shift, contrast_shift, gamma,
     titles=["original", "brightness shift", "contrast", "gamma"])

# table

for name, image in [("brightness", brightness_shift), ("contrast", contrast_shift), ("gamma", gamma)]:
    print(f"Name: `{name}`, Mean: {np.mean(image)} (diff: {(np.mean(image) - gray_mean):.4f}), Std Dev: {np.std(image)} (diff: {(np.std(image) - gray_std):.4f})")


In [ ]:
# QUESTION 3.2: \alpha=4 and histogram
contrast_shift_by_4 = np.clip(4 * (gray - 0.5) + 0.5, 0.0, 1.0)

show(gray, contrast_shift_by_4, titles=["original", "contrast shift by 4"])

# histogram
plt.figure()
plt.hist(gray.ravel(), bins=256, alpha=0.5, label="before") # slight transparency considered in alpha=0.5
plt.hist(contrast_shift_by_4.ravel(), bins=256, alpha=0.5, label="after")
plt.legend()
plt.show()


**Q3.2.** Check the claims of the book's §1.2 numerically: which of the three operations moved $\mu$ and left $\sigma$ alone, which scaled $\sigma$, and did any keep both? Where does clipping enter — push $\alpha$ to 4 and look at the histogram.

*Answer:* The brightness shift moved the mean by about $0.2$ and left the standard deviation almost unchanged. The contrast operation about the midpoint scaled the spread by roughly $1.3$ while keeping the center virtually unchaned, so it changed the standard deviation much more than the mean. The gamma operation is nonlinear, so it changed both the mean and the standard deviation. None kept both constant across the operation.

Clipping the brightened image saturates the white tones of the image, capping values above 1 to 1 (or 255, in the `unit8` case). For constrast, values that fall below 0 (the dark tones) and the values above 1 are capped at 1. For gamma, the stretch of midtones above 1 are capped at 1 as well.

When $\alpha$ is pushed to $4$, clipping at $0$ and $1$ truncates the histogram tails and the measured spread stops growing linearly, so the histogram visibly saturates at both ends.

---

**Q3.3.** Contrast stretch: map the image's actual range onto $[0,1]$ with $f(v) = (v - v_{\min}) / (v_{\max} - v_{\min})$. Apply it to a low-contrast version of the image (`0.3 * gray + 0.35`) and confirm it restores the full range.

In [ ]:
# YOUR CODE HERE

low_contrast = 0.3 * gray + 0.35
stretched = (low_contrast - low_contrast.min()) / (low_contrast.max() - low_contrast.min())

print("low contrast range:", low_contrast.min(), low_contrast.max())
print("stretched range:", stretched.min(), stretched.max())
print("Number of pixels with different values between original and stretched:", np.count_nonzero(gray - stretched > 1e-10))
show(gray, low_contrast, stretched, titles=["original", "low contrast", "contrast stretched"])


Given that the number of pixels that are different between the original and the constrast stretched are 0, this confirms that `gray` = `low_contrast`.

---

**Q3.4.** Histogram equalisation. The stretch fixes the range but not the shape; equalisation remaps values so the histogram becomes as flat as the data allows. The recipe: histogram the values into 256 bins, take the cumulative sum, normalise it to $[0,1]$ — that cumulative curve *is* the transfer function. Apply it with `np.interp`:

In [ ]:
hist, bin_edges = np.histogram(gray, bins=256, range=(0, 1))
cdf = hist.cumsum()
cdf = cdf / cdf[-1]
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
eq = np.interp(gray, bin_centers, cdf)

show(gray, eq, titles=["original grayscale", "equalised"])

# histograms
plt.figure(figsize=(8, 4))
plt.hist(gray.ravel(), bins=256, range=(0, 1), alpha=0.5, label="before")
plt.hist(eq.ravel(), bins=256, range=(0, 1), alpha=0.5, label="after")
plt.legend()
plt.show()


Display the image before and after, with both histograms. Where in the scene did equalisation spend its effort?

*Answer:* The histogram equalisation spent most of its effort in the regions that were previously crowded into a narrow range, specifically in the regions $[0.6, 1]$. As a result, the bright tones have been normalised highlighting features like the clouds in higher contrast.

---

## 4. Convolution from scratch

Everything after this point in the course — every filter in this lab, every layer of every network from Lab 2 onward — is this operation. Write it once yourself, by hand, so that it is never a mystery:

$$(I * K)(i, j) = \sum_{m}\sum_{n} I(i - m,\; j - n)\, K(m, n)$$

The index arithmetic ($i-m$, not $i+m$) means the kernel is *flipped* before it slides — that is what makes this convolution rather than correlation.

**Q4.1.** Implement `conv2(image, K)`: zero-pad the image so the output has the same height and width, flip the kernel, and slide. Two nested loops over output pixels, with a vectorised multiply-and-sum inside, is the intended solution — no `scipy` inside the function.


In [ ]:
def conv2(image, K):
    # suggested defensive programming
    if image.ndim != 2 or K.ndim != 2:
        raise ValueError("conv2 expects 2D arrays")
    if K.shape[0] % 2 == 0 or K.shape[1] % 2 == 0:
        raise ValueError("kernel must have odd dimensions")
    
    # firstly, we pad the image with a zero count from the kernel `K`'s shape
    h, w = image.shape
    kh, kw = K.shape
    pad_h, pad_w = kh // 2, kw // 2
    padded = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode="constant")
    K_flipped = K[::-1, ::-1] # we flip the kernel for convolution
    
    out = np.empty((h, w), dtype=float)

    for i in range(h):
        for j in range(w):
            patch = padded[i:i + kh, j:j + kw]
            out[i, j] = np.sum(patch * K_flipped)

    return out


In [ ]:
# Verification — run unchanged. Passes silently or fails loudly.
rng = np.random.default_rng(0)
A = rng.random((32, 32))
K = rng.random((5, 3))
assert np.allclose(conv2(A, K), ndimage.convolve(A, K, mode="constant")), \
    "conv2 does not match scipy — check padding and the kernel flip"
print("conv2 matches scipy.ndimage.convolve")

**Q4.2.** `ndimage.correlate` is the same sliding window without the flip. Apply both your `conv2` and `ndimage.correlate` to `gray` with the Sobel kernel below and compare the outputs. Why do symmetric kernels hide this difference, and what does that mean for a blur?

In [ ]:
sobel_x = np.array([[-1, 0, 1],
                    [-2, 0, 2],
                    [-1, 0, 1]], dtype=float)

# YOUR CODE HERE

conv_result = conv2(gray, sobel_x)
corr_result = ndimage.correlate(gray, sobel_x, mode="constant")

print("conv2 vs correlate max diff:", np.max(np.abs(conv_result - corr_result)))
show(conv_result, corr_result, titles=["conv2", "ndimage.correlate"])


*Answer:* `conv2` implements convolution, while `ndimage.correlate` is correlation. They differ whenever the kernel is not symmetric under reflection; the Sobel operator is not symmetric, so the outputs are not identical. Symmetric kernels, such as box or Gaussian blur kernels, hide this difference because flipping them leaves them unchanged, which is why a blur looks the same whether you use convolution or correlation.

---

**Q4.3.** Build a normalised $5\times5$ box kernel and a $31\times31$ Gaussian kernel with $\sigma = 5$ (construct the 1D Gaussian from its formula, then take the outer product). Blur `gray` with each — `ndimage.convolve` is allowed from here on, your `conv2` has earned it — and display the results side by side. Where do the box's streaky artefacts show?

In [ ]:
# YOUR CODE HERE

box_kernel = np.ones((5, 5), dtype=float) / 25.0 # normalised by 25 (number of elements in the kernel)
x = np.arange(-(31 // 2), 31 // 2 + 1)
g1d = np.exp(-(x ** 2) / (2 * 5 ** 2))
g1d = g1d / g1d.sum()
gauss_kernel = np.outer(g1d, g1d)

box_blur = ndimage.convolve(gray, box_kernel, mode="nearest")
gauss_blur = ndimage.convolve(gray, gauss_kernel, mode="nearest")
show(box_blur, gauss_blur, titles=["5x5 box blur", "31x31 Gaussian blur"])


*Answer:* The box blur shows streaky artefacts around clear edges such as the outline of the bark of the tree and the shadows.

---

**Q4.4.** Separability. The Gaussian factors: filtering rows with the 1D kernel, then columns, equals the 2D convolution. Verify the outputs match (`np.allclose`), then time the two with `time.perf_counter` — average over a few runs. The operation-count argument predicts $961/62 \approx 15\times$; what do you measure, and why might the wall-clock ratio differ?

In [ ]:
# YOUR CODE HERE

full_2d = ndimage.convolve(gray, gauss_kernel, mode="nearest")
rows_filtered = ndimage.convolve(gray, g1d[None, :], mode="nearest")
sep_blur = ndimage.convolve(rows_filtered, g1d[:, None], mode="nearest")

print("separable matches full 2D:", np.allclose(sep_blur, full_2d))

n_runs = 5
times_full = []
times_sep = []

for _ in range(n_runs):
    t0 = time.perf_counter()
    ndimage.convolve(gray, gauss_kernel, mode="nearest")
    times_full.append(time.perf_counter() - t0)
    t0 = time.perf_counter()
    rows_filtered = ndimage.convolve(gray, g1d[None, :], mode="nearest")
    ndimage.convolve(rows_filtered, g1d[:, None], mode="nearest")
    times_sep.append(time.perf_counter() - t0)

print("average full 2D time:", np.mean(times_full))
print("average separable time:", np.mean(times_sep))
print("speedup:", np.mean(times_full) / np.mean(times_sep))


*Answer:* The separable implementation reproduces the full 2D Gaussian blur up to floating-point rounding, and it is much faster. The arithmetic argument predicts about a $15\times$ speedup for a $31	\times 31$ kernel, and the measured speedup is usually close to that even though the wall-clock ratio is a bit smaller because of computer overhead from repeated array passes and memory traffic.

---

**Q4.5.** Gradients. Convolve `gray` with `sobel_x` and its transpose to get $g_x$ and $g_y$, then compute the gradient magnitude $\sqrt{g_x^2 + g_y^2}$ and orientation (`np.arctan2(gy, gx)`). Display the magnitude (rescale it into $[0,1]$ for display). Which edges in the scene answer to $g_x$ but not $g_y$?


In [ ]:
# YOUR CODE HERE

gx = conv2(gray, sobel_x)
gy = conv2(gray, sobel_x.T)
magnitude = np.sqrt(gx ** 2 + gy ** 2)
magnitude_disp = (magnitude - magnitude.min()) / (magnitude.max() - magnitude.min())
orientation = np.arctan2(gy, gx)

show(magnitude_disp, titles=["gradient magnitude"])
print("orientation range:", orientation.min(), orientation.max())


*Answer:* The gradient magnitude highlights the strongest boundaries in the scene, especially the object outlines. The $x$-Sobel responds strongly to vertical edges (that is, they have a strong horizontal change), while the $y$-Sobel responds to edges with strong vertical change (that is, the horizontal edges).

---

## 5. The noise film-strip

Additive Gaussian noise perturbs every pixel independently:

$$x_{ij} \leftarrow x_{ij} + \varepsilon, \qquad \varepsilon \sim N(0, \sigma^2)$$

Against values in $[0,1]$, $\sigma$ has direct meaning: at $\sigma = 0.05$ a typical pixel is off by a twentieth of the full range; at $\sigma = 0.5$ the noise is as strong as the signal.

**Q5.1.** Implement `add_noise(image, sigma, rng)`. Do **not** clip the result — keep the true noisy values and let `show` clip for display only. Build the film-strip: `gray` at the six waypoints $\sigma \in \{0,\ 0.05,\ 0.1,\ 0.2,\ 0.5,\ 1.0\}$, displayed as one row, clean to pure static, each panel titled with its $\sigma$.

In [ ]:
WAYPOINTS = [0.0, 0.05, 0.1, 0.2, 0.5, 1.0]

def add_noise(image, sigma, rng):
    # YOUR CODE HERE
    return image + rng.normal(0.0, sigma, size=image.shape)

# YOUR CODE HERE — build and display the strip
 
noisy_images = [add_noise(gray, s, np.random.default_rng(1)) for s in WAYPOINTS]
show(*noisy_images, titles=[f"$\\sigma = {s}$" for s in WAYPOINTS])


**Q5.2.** Noise adds variance: $\mathrm{Var}(x + \varepsilon) = \mathrm{Var}(x) + \sigma^2$. Verify this numerically at each waypoint. The signal-to-noise ratio in decibels is $10 \log_{10}(\mathrm{Var}(x) / \sigma^2)$ — report it for each waypoint too.

In [ ]:
# YOUR CODE HERE

signal_var = np.var(gray)

for sigma in WAYPOINTS:
    noisy = add_noise(gray, sigma, np.random.default_rng(2))
    noisy_var = np.var(noisy)
    snr_db = float("inf") if sigma == 0 else 10 * np.log10(signal_var / (sigma ** 2))
    print(f"sigma={sigma:.2f}: var(noisy)={noisy_var:.4f}, expected={signal_var + sigma ** 2:.4f}, SNR={snr_db:.2f} dB")


Save the strip — the array, not just the picture:

In [ ]:
strip = np.stack([add_noise(gray, s, np.random.default_rng(1)) for s in WAYPOINTS])
np.savez("filmstrip.npz", sigmas=np.array(WAYPOINTS), strip=strip)
!ls -l filmstrip.npz

The strip is this lab's test bench — Sections 6 and 7 run the chapter's detectors down it. It is also the first piece of something larger: **keep `filmstrip.npz` somewhere durable** (download it, or copy it into your Drive). A later lab asks for the file, and for the `add_noise` function that made it.

In [ ]:
from google.colab import files
files.download("filmstrip.npz")

## 6. Edges on the strip

How does Canny behave as the noise rises? `skimage.feature.canny` runs the whole pipeline — Gaussian smooth, gradient, non-maximum suppression, hysteresis — with the smoothing scale as `sigma` (this is a *different* $\sigma$ from the noise level; keep the two apart).

In [ ]:
from skimage.feature import canny

NOISE = [0.0, 0.02, 0.05, 0.08, 0.1, 0.15, 0.2, 0.3, 0.5]

**Q6.1.** For each noise level in `NOISE`, run `canny(noisy, sigma=2)` and count the edge pixels (`edges.sum()`). Plot the count against noise level. The count *rises*. Look at the edge maps for $\sigma = 0$ and $\sigma = 0.3$ side by side and explain why counting edge pixels cannot measure edge survival.

In [ ]:
# YOUR CODE HERE

from skimage.feature import canny

edge_counts = []
edge_maps = []
for noise_level in NOISE:
    noisy = add_noise(gray, noise_level, np.random.default_rng(3))
    edges = canny(noisy, sigma=2)
    edge_counts.append(edges.sum())
    edge_maps.append(edges)

plt.figure(figsize=(8, 4))
plt.plot(NOISE, edge_counts, marker="o")
plt.xlabel("noise level")
plt.ylabel("edge pixel count")
plt.show()

clean_edges = canny(gray, sigma=2)
noisy_edges = canny(add_noise(gray, 0.3, np.random.default_rng(4)), sigma=2)
show(clean_edges, noisy_edges, titles=["sigma=0", "sigma=0.3"])


*Answer:* The edge-count curve rises because more noise creates more weak gradient responses that survive the Canny threshold. That does not mean the detector is preserving the original edges better; it often means it is adding many false positives, especially in textured regions.

---

One tool you have not met yet: **dilation**. On a binary image, one step of dilation turns on every pixel that touches a `True` pixel, so a shape grows one pixel fatter all round. The cell below shows it on a toy array — three pixels become a thickened bar:

In [ ]:
toy = np.zeros((5, 7), dtype=bool)
toy[2, 2:5] = True
print(toy.astype(int))
print()
print(ndimage.binary_dilation(toy).astype(int))

**Q6.2.** A metric that can: **precision against the clean edges**. Take the clean edge map and dilate it by one pixel, so that a detection landing one pixel off a clean edge still counts. Then measure the fraction of *detected* edge pixels that land inside the dilated map:

$$\text{precision} = \frac{|\,\text{detected} \cap \text{dilated clean}\,|}{|\,\text{detected}\,|}$$

Compute precision across `NOISE` for `canny` at smoothing $\sigma \in \{1, 2, 4\}$ (each against its own clean map) and plot the three curves on one set of axes.

In [ ]:
# YOUR CODE HERE

precisions = {sigma: [] for sigma in [1, 2, 4]}

for smooth_sigma in [1, 2, 4]:
    clean_edges = canny(gray, sigma=smooth_sigma)
    dilated_clean = ndimage.binary_dilation(clean_edges)
    for noise_level in NOISE:
        noisy = add_noise(gray, noise_level, np.random.default_rng(5))
        detected = canny(noisy, sigma=smooth_sigma)
        if detected.sum() == 0:
            precisions[smooth_sigma].append(0.0)
        else:
            precision = np.sum(detected & dilated_clean) / detected.sum()
            precisions[smooth_sigma].append(precision)

plt.figure(figsize=(8, 4))

for smooth_sigma in [1, 2, 4]:
    plt.plot(NOISE, precisions[smooth_sigma], marker="o", label=f"sigma={smooth_sigma}")
    
plt.xlabel("noise level")
plt.ylabel("precision")
plt.legend()
plt.show()


**Q6.3.** Read your plot. Which curve collapses first? Smoothing more delays the collapse — what does $\sigma = 4$ cost on the *clean* image? (Display the three clean edge maps.) One caution before crediting $\sigma = 1$ with its floor of ${\sim}0.6$: what fraction of the whole image is within one pixel of a $\sigma = 1$ clean edge, and what does that make the chance level of this metric?

In [ ]:
# YOUR CODE HERE

for smooth_sigma in [1, 2, 4]:
    clean_edges = canny(gray, sigma=smooth_sigma)
    print(f"sigma={smooth_sigma}: clean edge fraction within one pixel = {ndimage.binary_dilation(clean_edges).mean():.3f}")
    show(clean_edges, titles=[f"clean edges, sigma={smooth_sigma}"])


*Answer:* The curve with smaller smoothing, $\sigma = 1$, collapses first because it follows the noise more closely. Larger smoothing, such as $\sigma = 4$, is more robust but also blurs the clean edge map, making the clean edges thicker and the metric less sensitive to fine detail. The chance level of this metric is roughly the fraction of the image within one pixel of a clean edge which is $0.584$, so a precision score above that baseline is meaningful only when the detector clearly exceeds it.

---

## 7. SIFT on the strip

SIFT finds keypoints and describes their neighbourhoods so that the same physical point can be matched between two photographs — across rotation and scale. The book's matching figure pairs the scene with a rotated, rescaled copy of itself; do the same, then let the copy degrade.

In [ ]:
from skimage.feature import SIFT, match_descriptors
from skimage.transform import rescale, rotate

view2 = rescale(rotate(gray, 22, resize=True), 0.75)
show(gray, view2, titles=["view 1", "view 2: rotated 22°, rescaled 0.75"])

**Q7.1.** Detect and match. `SIFT().detect_and_extract(image)` fills `.keypoints` and `.descriptors`; `match_descriptors(d1, d2, max_ratio=0.72, cross_check=True)` applies the ratio test both ways. How many keypoints on each view, and how many matches survive between the clean pair?

In [ ]:
# YOUR CODE HERE

try:
    # emulate and keep order as is - scikit-image gave me a hard time here. Solution in docs
    descriptor_extractor = SIFT()

    descriptor_extractor.detect_and_extract(gray)
    gray_kp = descriptor_extractor.keypoints
    gray_desc = descriptor_extractor.descriptors

    descriptor_extractor.detect_and_extract(view2)
    view2_kp = descriptor_extractor.keypoints
    view2_desc = descriptor_extractor.descriptors

    if gray_kp is None or view2_kp is None:
        raise RuntimeError("SIFT did not return keypoints/descriptors")

    matches = match_descriptors(gray_desc, view2_desc, max_ratio=0.72, cross_check=True)
    print("keypoints view1:", len(gray_kp))
    print("keypoints view2:", len(view2_kp))
    print("matches:", len(matches))
except Exception as exc:
    print("SIFT detection failed:", exc)
    print("keypoints view1: 0")
    print("keypoints view2: 0")
    print("matches: 0")

**Q7.2.** Now run the noise down the strip: for each level in `NOISE`, add noise to `view2` (clip to $[0,1]$ this time — SIFT expects an image), re-detect, re-match against the *clean* first view, and record the match count. Plot match count against noise level. Wrap `detect_and_extract` in `try/except RuntimeError` and record zero if it fails — at high noise there may be nothing left to detect.

In [ ]:
# YOUR CODE HERE

match_counts = []

for noise_level in NOISE:
    noisy_view2 = np.clip(add_noise(view2, noise_level, np.random.default_rng(6)), 0.0, 1.0)

    try:
        descriptor_extractor = SIFT()
        descriptor_extractor.detect_and_extract(noisy_view2)
        kp_noisy = descriptor_extractor.keypoints
        desc_noisy = descriptor_extractor.descriptors

        if kp_noisy is None:
            raise RuntimeError("SIFT found no keypoints")

        matches = match_descriptors(view2_desc, desc_noisy, max_ratio=0.72, cross_check=True)
        match_counts.append(len(matches))
    except Exception:
        match_counts.append(0)

plt.figure(figsize=(8, 4))
plt.plot(NOISE, match_counts, marker="o")
plt.xlabel("noise level")
plt.ylabel("match count")
plt.show()


**Q7.3.** At what noise level have half the clean pair's matches gone? Compare with the Canny curves of Q6.2: is descriptor matching more or less robust to this noise than edge detection, and why might averaging over a $16\times16$ neighbourhood help?

*Answer:* Using the curve, the half-match point is roughly $0.06$. Compared with the Canny precision curves, SIFT matching is more robust because each descriptor summarises a local patch rather than a single thresholded edge. Averaging over a $16 \times 16$ neighbourhood makes the descriptor less sensitive to a few noisy pixels.

---

## 8. Textons and Bag-of-Visual-Words

The chapter's texture pipeline, end to end, on the book's bark and grass: *filter* with a bank, *describe* each pixel by its responses, *cluster* the descriptions into a vocabulary of textons, *count* the vocabulary — then classify.


In [ ]:
from scipy.cluster.vq import kmeans2
from skimage.filters import gabor_kernel

bark = io.imread("bark.png").astype(np.float64) / 255.0
grass = io.imread("grass.png").astype(np.float64) / 255.0
show(bark, grass, titles=["bark", "grass"])

**Q8.1.** Build the bank: Gabor kernels at three frequencies $f \in \{0.1, 0.2, 0.4\}$ and four orientations $\theta \in \{0°, 45°, 90°, 135°\}$ — twelve kernels. `gabor_kernel(frequency, theta=...)` returns a complex kernel; display the real parts as one $3\times4$ grid of images. Match what you see against the Gabor formula: which way does $\theta$ turn the stripes, and what does $f$ change?

In [ ]:
# YOUR CODE HERE

frequencies = [0.1, 0.2, 0.4]
orientations = np.deg2rad([0, 45, 90, 135])
kernels = []

for freq in frequencies:
    for theta in orientations:
        kernels.append(gabor_kernel(frequency=freq, theta=theta))

fig, axes = plt.subplots(3, 4, figsize=(12, 8))
axes = axes.ravel()

for ax, kernel in zip(axes, kernels):
    ax.imshow(np.real(kernel), cmap="gray") # real parts only
    ax.set_axis_off()

plt.tight_layout()
plt.show()


*Answer:* The Gabor filters rotate with $\theta$, so changing the orientation turns the stripes left or right. Increasing the frequency $f$ makes the stripes closer together and therefore makes the filter respond to finer texture detail.

---

**Q8.2.** Filter. For each kernel, the response magnitude is $\sqrt{(I * \mathrm{Re}\,K)^2 + (I * \mathrm{Im}\,K)^2}$ — convolve with the real and imaginary parts separately (`ndimage.convolve`, `mode="nearest"`) and combine. Write `responses(image)` returning an `H × W × 12` array, and run it on both textures. Display a few response channels: which kernels fire on bark, which on grass?

In [ ]:
def responses(image):
    if image.ndim == 3:
        image = color.rgb2gray(image)

    H, W = image.shape
    response = np.empty((H, W, 12), dtype=float)

    for k, kernel in enumerate(kernels):
        real_part = np.real(kernel)
        imag_part = np.imag(kernel)
        real_resp = ndimage.convolve(image, real_part, mode="nearest")
        imag_resp = ndimage.convolve(image, imag_part, mode="nearest")
        response[:, :, k] = np.sqrt(real_resp ** 2 + imag_resp ** 2)

    return response

bark_resp = responses(bark)
grass_resp = responses(grass)
print("bark response shape:", bark_resp.shape)
print("grass response shape:", grass_resp.shape)

fig, axes = plt.subplots(4, 6, figsize=(18, 10))
axes = axes.ravel()

for idx in range(12):
    ax = axes[idx]
    ax.imshow(bark_resp[:, :, idx], cmap="gray")
    ax.set_title(f"bark {idx}", fontsize=8)
    ax.set_axis_off()

for idx in range(12, 24):
    ax = axes[idx]
    ax.imshow(grass_resp[:, :, idx - 12], cmap="gray")
    ax.set_title(f"grass {idx - 12}", fontsize=8)
    ax.set_axis_off()

plt.tight_layout()
plt.show()


*Answer:* The grass responds strongly to filters oriented at $45^\circ$ and $135^\circ$ at increasing frequencies, and the bark responds strongly to filters $0^\circ$ at increasing frequencies as well. 

---

**Q8.3.** Cluster. Each pixel is now a 12-dimensional vector. Use only the **left half** (columns 0–63) of each texture — the right halves stay unseen for testing. Pool the left-half pixels of both textures, sample 4 000 of them, and cluster with `kmeans2(sample, k=8, seed=3, minit="++")`. The eight centroids are the texton vocabulary.


In [ ]:
# YOUR CODE HERE

bark_left = bark_resp[:, :64, :]
grass_left = grass_resp[:, :64, :]
sample = np.vstack([bark_left.reshape(-1, 12), grass_left.reshape(-1, 12)])
sample = sample[np.random.default_rng(0).choice(sample.shape[0], size=min(4000, sample.shape[0]), replace=False)]
centroids, labels = kmeans2(sample, k=8, minit="++", seed=3)
print("centroids shape:", centroids.shape)


**Q8.4.** Map and count. Assign every pixel of each texture to its nearest texton and display the two texton maps (`imshow` with `cmap="tab10"`). Then write `texton_hist(patch_responses)`: nearest-texton label per pixel, count into 8 bins, normalise to sum to 1. Plot the histogram of each texture's left half as a bar chart — the two signatures the classifier will lean on.

In [ ]:
# YOUR CODE HERE

def assign_textons(response_map, centroids):
    flat = response_map.reshape(-1, 12)
    diffs = flat[:, None, :] - centroids[None, :, :]
    dists = np.sqrt(np.sum(diffs ** 2, axis=2))
    return np.argmin(dists, axis=1).reshape(response_map.shape[:2])

def texton_hist(patch_responses):
    labels = assign_textons(patch_responses, centroids)
    hist = np.bincount(labels.ravel(), minlength=8).astype(float)
    hist /= hist.sum()
    return hist

bark_labels = assign_textons(bark_resp, centroids)
grass_labels = assign_textons(grass_resp, centroids)
show(bark_labels, grass_labels, titles=["bark texton map", "grass texton map"], cmap="tab10")

bark_hist = texton_hist(bark_resp[:, :64, :])
grass_hist = texton_hist(grass_resp[:, :64, :])

plt.figure(figsize=(8, 4))
plt.bar(np.arange(8), bark_hist, alpha=0.5, label="bark")
plt.bar(np.arange(8), grass_hist, alpha=0.5, label="grass")
plt.legend()
plt.show()


**Q8.5.** Classify. Cut each half into $32\times32$ crops — eight training crops per texture from the left halves, eight test crops per texture from the right. Histogram every crop; classify each test crop by its nearest training histogram under the $\chi^2$ distance

$$\chi^2(h, g) = \tfrac{1}{2}\sum_k \frac{(h_k - g_k)^2}{h_k + g_k + \epsilon}$$

and report accuracy out of 16, with the label of any crop that misses.

In [ ]:
# YOUR CODE HERE

def make_crops(response_map, crop_size=32, step=16):
    crops = []
    for y in range(0, response_map.shape[0] - crop_size + 1, step):
        for x in range(0, response_map.shape[1] - crop_size + 1, step):
            crops.append(response_map[y:y + crop_size, x:x + crop_size, :])
            
    return crops

def chi2(h, g):
    return 0.5 * np.sum(((h - g) ** 2) / (h + g + 1e-12))

bark_train = make_crops(bark_resp[:, :64, :])[:8]
grass_train = make_crops(grass_resp[:, :64, :])[:8]
bark_test = make_crops(bark_resp[:, 64:, :])[:8]
grass_test = make_crops(grass_resp[:, 64:, :])[:8]

train_hists = [texton_hist(crop) for crop in bark_train + grass_train]
train_labels = ["bark"] * 8 + ["grass"] * 8

predictions = []
true_labels = ["bark"] * 8 + ["grass"] * 8

for crop in bark_test + grass_test:
    h = texton_hist(crop)
    dists = [chi2(h, t) for t in train_hists]
    pred = train_labels[np.argmin(dists)]
    predictions.append(pred)

accuracy = np.mean([p == t for p, t in zip(predictions, true_labels)])
print("accuracy:", accuracy)

for idx, (pred, true) in enumerate(zip(predictions, true_labels)):
    if pred != true:
        print("missed crop", idx, "predicted", pred, "true", true)


**Q8.6.** The histogram threw away *where* every texton fell. For texture, that loss is the point — say why. Name one recognition task from the chapter where discarding all spatial layout would be fatal.

*Answer:* The histogram-based classifier works because the texton histograms capture the texture distribution well, so the test crops usually land close to the correct training signature. Any misses are the result of a crop whose local texton distribution is not representative of its class.

## 9. Before you submit

- [ ] **Runtime ▸ Restart session and run all** — then read every output, top to bottom.
- [ ] Group members filled in at the top; every member can explain every cell.
- [ ] Every *Answer:* cell has an answer in it.
- [ ] `filmstrip.npz` is saved somewhere you will find it later in the semester.
- [ ] **File ▸ Download ▸ Download .ipynb**, and one member submits it on Moodle before **Thursday 30 July, 17:00**
